# 00 — `datasets` & tokenizers

Loading text corpora with **🤗 `datasets`** and turning text into model inputs with a
**`transformers` tokenizer** (including chat templates and the reasoning *thinking*
toggle). These two primitives underpin every other notebook — generation, training, and
evaluation all start here.

**Library focus:** `datasets.load_dataset`, `transformers.AutoTokenizer`.

In [ ]:
MODEL = "Qwen/Qwen3.5-4B"  # research model; swap for a smaller id for a quick run

## Load a corpus from the Hub

`load_dataset(repo, name=None, split=...)` returns a `Dataset`; index a column by name to
get a list of strings. `.select(range(n))` slices *before* materialising the column.

In [ ]:
from datasets import load_dataset

# General-text corpus used as the "real general" replay set in the research program.
ds = load_dataset("NeelNanda/pile-10k", split="train")
ds = ds.select(range(8))  # keep it tiny for the demo
texts = ds["text"]
print(f"{len(ds)} rows; columns = {ds.column_names}")
print(texts[0][:200], "...")

A dataset that needs a config name (and a different text field) — the medical domain
corpus. The spec mirrors `llm_core.corpus`'s `hf:<dataset>:<split>:<field>` strings.

In [ ]:
domain = load_dataset("MedRAG/textbooks", split="train").select(range(4))
print(domain.column_names)  # -> the 'content' field holds the text
print(domain["content"][0][:200], "...")

## Load local files (`.jsonl` / `.txt`)

`datasets` reads local files directly; `.txt` (one text per line) and `.jsonl` (pick a
field) are the two formats the generation/validation tooling writes under `runs/`.

In [ ]:
import json
import pathlib

tmp = pathlib.Path("/workspaces/llm-research/runs/_examples_00")
tmp.mkdir(parents=True, exist_ok=True)
(tmp / "corpus.txt").write_text("first document\nsecond document\n")
(tmp / "corpus.jsonl").write_text(
    "\n".join(json.dumps({"text": t}) for t in ["alpha", "beta", "gamma"])
)

txt = load_dataset("text", data_files=str(tmp / "corpus.txt"), split="train")
jsonl = load_dataset("json", data_files=str(tmp / "corpus.jsonl"), split="train")
print("txt :", txt["text"])
print("json:", jsonl["text"])

## Tokenize text

`AutoTokenizer` loads the model's tokenizer. Encoding returns `input_ids` (+ an
`attention_mask`); decoding inverts it. Set `pad_token` to `eos_token` when the model
has none — required before batched generation or training.

In [ ]:
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained(MODEL)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

enc = tok("The capital of France is", return_tensors="pt")
print("input_ids:", enc["input_ids"])
print("decoded  :", tok.decode(enc["input_ids"][0]))

## Chat templates & the *thinking* toggle

Instruct models ship a **chat template** that formats a message list into the exact prompt
string the model expects. `add_generation_prompt=True` appends the assistant turn marker
so the model continues as the assistant.

Reasoning models accept an `enable_thinking` kwarg. For answer-extraction tasks the repo
turns it **off** — a verbose reasoning preamble otherwise overruns the answer budget (a
real lesson: GSM8K scored 0.00 with thinking on vs 0.60 off). Pass the kwarg only when the
template supports it (older templates raise `TypeError`).

In [ ]:
messages = [{"role": "user", "content": "Name three primary colors."}]

prompt = tok.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)
print(repr(prompt[:300]))

In [ ]:
def render_chat(tokenizer, content, thinking=False):
    msgs = [{"role": "user", "content": content}]
    kwargs = {"add_generation_prompt": True, "tokenize": False}
    try:
        return tokenizer.apply_chat_template(msgs, enable_thinking=thinking, **kwargs)
    except TypeError:
        return tokenizer.apply_chat_template(msgs, **kwargs)  # template w/o the toggle


print(render_chat(tok, "Name three primary colors.")[:300])

> **Project pointer:** `llm_core.corpus.load_corpus` wraps the `hf:`/`.txt`/`.jsonl`
> loading shown here; `llm_core.models` (`resolve_profile` / `ModelProfile.render_chat`)
> auto-detects chat-template + thinking support so callers don't hand-roll the try/except.